# HDB join and overlap EDA

This notebook audits a direct HDB left join on normalized `BLOCK + STREET`. It deliberately does **not** freeze the lookup to locations seen in training and does not impute missing values. It also measures individual and composite overlap among train, test, and `sg-hdb-block.csv`.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
project_root = next((root for root in roots if (root / 'data' / 'train.csv').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')

data_dir = project_root / 'data'
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
hdb = pd.read_csv(data_dir / 'auxiliary' / 'sg-hdb-block.csv')
pd.DataFrame({
    'dataset': ['train', 'test', 'hdb'],
    'rows': [len(train), len(test), len(hdb)],
    'columns': [train.shape[1], test.shape[1], hdb.shape[1]],
})

,dataset,rows,columns
0,train,150000,11
1,test,50000,10
2,hdb,9464,11


In [2]:
# Match the project's categorical-key cleaning while keeping the join itself simple.
def normalize_location_key(series):
    return (
        series.astype('string')
        .str.lower()
        .str.strip()
        .str.replace(r'[-_/]+', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

LOCATION_COLUMNS = ['TOWN', 'BLOCK', 'STREET']
for frame in (train, test, hdb):
    for column in LOCATION_COLUMNS:
        frame[f'{column}_KEY'] = normalize_location_key(frame[column])

KEY_COLUMNS = ['TOWN_KEY', 'BLOCK_KEY', 'STREET_KEY']
pd.concat([
    train[LOCATION_COLUMNS + KEY_COLUMNS].head(3).assign(dataset='train'),
    test[LOCATION_COLUMNS + KEY_COLUMNS].head(3).assign(dataset='test'),
    hdb[LOCATION_COLUMNS + KEY_COLUMNS].head(3).assign(dataset='hdb'),
], ignore_index=True)

,TOWN,BLOCK,STREET,TOWN_KEY,BLOCK_KEY,STREET_KEY,dataset
0,punggol,604a,punggol road,punggol,604a,punggol road,train
1,sengkang,290c,compassvale crescent,sengkang,290c,compassvale crescent,train
2,queenstown,5,ghim moh road,queenstown,5,ghim moh road,train
3,kallang/whampoa,74,Whampoa Drive,kallang whampoa,74,whampoa drive,test
4,bukit timah,6,farrer road,bukit timah,6,farrer road,test
5,pasir ris,755,Pasir Ris Street 71,pasir ris,755,pasir ris street 71,test
6,serangoon,106,serangoon north avenue 1,serangoon,106,serangoon north avenue 1,hdb
7,bukit merah,34,telok blangah way,bukit merah,34,telok blangah way,hdb
8,punggol,226b,sumang lane,punggol,226b,sumang lane,hdb


In [3]:
# A many-to-one join is safe only if BLOCK + STREET uniquely identifies an HDB row.
JOIN_KEYS = ['BLOCK_KEY', 'STREET_KEY']
HDB_VALUES = ['LATITUDE', 'LONGITUDE', 'MAX_FLOOR', 'YEAR_COMPLETED', 'REGION']
hdb_key_counts = hdb.groupby(JOIN_KEYS, dropna=False).size().rename('rows').reset_index()
duplicate_hdb_keys = hdb_key_counts.loc[hdb_key_counts['rows'] > 1]
hdb_key_integrity = pd.DataFrame([{
    'hdb_rows': len(hdb),
    'unique_block_street_keys': len(hdb_key_counts),
    'duplicate_keys': len(duplicate_hdb_keys),
    'extra_duplicate_rows': int((duplicate_hdb_keys['rows'] - 1).sum()),
}])
display(hdb_key_integrity)
display(duplicate_hdb_keys.head(20))

,hdb_rows,unique_block_street_keys,duplicate_keys,extra_duplicate_rows
0,9464,9464,0,0


,BLOCK_KEY,STREET_KEY,rows


In [4]:
# Direct left join: no train-only lookup restriction and no fallback imputation.
hdb_lookup = hdb[JOIN_KEYS + HDB_VALUES].drop_duplicates(JOIN_KEYS)

def direct_hdb_join(frame):
    return frame.merge(
        hdb_lookup,
        on=JOIN_KEYS,
        how='left',
        validate='many_to_one',
        indicator='_hdb_join',
    )

train_hdb = direct_hdb_join(train)
test_hdb = direct_hdb_join(test)

join_rows = []
for dataset_name, original, joined in [
    ('train', train, train_hdb),
    ('test', test, test_hdb),
]:
    missing_match = joined['_hdb_join'].eq('left_only')
    missing_any_value = joined[HDB_VALUES].isna().any(axis=1)
    join_rows.append({
        'dataset': dataset_name,
        'input_rows': len(original),
        'joined_rows': len(joined),
        'matched_rows': int((~missing_match).sum()),
        'missing_join_rows': int(missing_match.sum()),
        'missing_join_pct': missing_match.mean(),
        'rows_missing_any_HDB_value': int(missing_any_value.sum()),
    })

direct_join_summary = pd.DataFrame(join_rows)
display(direct_join_summary.style.format({'missing_join_pct': '{:.3%}'}))
assert len(train_hdb) == len(train) and len(test_hdb) == len(test)
display(train_hdb.loc[train_hdb['_hdb_join'].eq('left_only'), LOCATION_COLUMNS].drop_duplicates().head(25))
display(test_hdb.loc[test_hdb['_hdb_join'].eq('left_only'), LOCATION_COLUMNS].drop_duplicates().head(25))

,dataset,input_rows,joined_rows,matched_rows,missing_join_rows,missing_join_pct,rows_missing_any_HDB_value
0,train,150000,150000,150000,0,0.000%,0
1,test,50000,50000,50000,0,0.000%,0


,TOWN,BLOCK,STREET


,TOWN,BLOCK,STREET


In [5]:
def unique_values(frame, columns):
    clean = frame[columns].dropna().drop_duplicates()
    if len(columns) == 1:
        return set(clean.iloc[:, 0].tolist())
    return set(clean.itertuples(index=False, name=None))

def directed_overlap(left_name, left, right_name, right, columns):
    left_values = unique_values(left, columns)
    right_values = unique_values(right, columns)
    overlap = left_values & right_values
    return {
        'key': ' + '.join(column.removesuffix('_KEY') for column in columns),
        'source': left_name,
        'comparison': right_name,
        'source_unique': len(left_values),
        'comparison_unique': len(right_values),
        'overlap_unique': len(overlap),
        'source_covered_by_comparison_pct': (
            len(overlap) / len(left_values) if left_values else np.nan
        ),
        'source_only_unique': len(left_values - right_values),
    }

frames = {'train': train, 'test': test, 'hdb': hdb}
individual_rows = []
for column in ['TOWN_KEY', 'STREET_KEY', 'BLOCK_KEY']:
    for left_name, right_name in [
        ('train', 'test'), ('test', 'train'),
        ('train', 'hdb'), ('test', 'hdb'),
    ]:
        individual_rows.append(directed_overlap(
            left_name, frames[left_name], right_name, frames[right_name], [column]
        ))

individual_overlap = pd.DataFrame(individual_rows)
display(individual_overlap.style.format({'source_covered_by_comparison_pct': '{:.3%}'}))

,key,source,comparison,source_unique,comparison_unique,overlap_unique,source_covered_by_comparison_pct,source_only_unique
0,TOWN,train,test,26,26,26,100.000%,0
1,TOWN,test,train,26,26,26,100.000%,0
2,TOWN,train,hdb,26,26,26,100.000%,0
3,TOWN,test,hdb,26,26,26,100.000%,0
4,STREET,train,test,576,573,573,99.479%,3
5,STREET,test,train,573,576,573,100.000%,0
6,STREET,train,hdb,576,576,576,100.000%,0
7,STREET,test,hdb,573,576,573,100.000%,0
8,BLOCK,train,test,2720,2660,2649,97.390%,71
9,BLOCK,test,train,2660,2720,2649,99.586%,11


In [6]:
# Composite keys distinguish blocks with the same number on different streets.
composite_rows = []
for columns in [
    ['BLOCK_KEY', 'STREET_KEY'],
    ['TOWN_KEY', 'BLOCK_KEY', 'STREET_KEY'],
]:
    for left_name, right_name in [
        ('train', 'test'), ('test', 'train'),
        ('train', 'hdb'), ('test', 'hdb'),
    ]:
        composite_rows.append(directed_overlap(
            left_name, frames[left_name], right_name, frames[right_name], columns
        ))

composite_overlap = pd.DataFrame(composite_rows)
display(composite_overlap.style.format({'source_covered_by_comparison_pct': '{:.3%}'}))

,key,source,comparison,source_unique,comparison_unique,overlap_unique,source_covered_by_comparison_pct,source_only_unique
0,BLOCK + STREET,train,test,9406,8940,8890,94.514%,516
1,BLOCK + STREET,test,train,8940,9406,8890,99.441%,50
2,BLOCK + STREET,train,hdb,9406,9464,9406,100.000%,0
3,BLOCK + STREET,test,hdb,8940,9464,8940,100.000%,0
4,TOWN + BLOCK + STREET,train,test,9406,8940,8890,94.514%,516
5,TOWN + BLOCK + STREET,test,train,8940,9406,8890,99.441%,50
6,TOWN + BLOCK + STREET,train,hdb,9406,9464,9406,100.000%,0
7,TOWN + BLOCK + STREET,test,hdb,8940,9464,8940,100.000%,0


In [7]:
# Test locations absent from train can still have valid direct HDB matches.
train_locations = train[['TOWN_KEY', 'BLOCK_KEY', 'STREET_KEY']].drop_duplicates()
test_locations = test[
    LOCATION_COLUMNS + ['TOWN_KEY', 'BLOCK_KEY', 'STREET_KEY']
].drop_duplicates(['TOWN_KEY', 'BLOCK_KEY', 'STREET_KEY'])
test_only_locations = test_locations.merge(
    train_locations.assign(seen_in_train=True),
    on=['TOWN_KEY', 'BLOCK_KEY', 'STREET_KEY'],
    how='left',
).loc[lambda frame: frame['seen_in_train'].isna()].drop(columns='seen_in_train')
test_only_locations = test_only_locations.merge(
    hdb_lookup.assign(has_hdb_match=True),
    on=JOIN_KEYS,
    how='left',
    validate='one_to_one',
)
print(f'Test-only TOWN + BLOCK + STREET locations: {len(test_only_locations):,}')
print(f'Of these, direct HDB matches: {test_only_locations.has_hdb_match.notna().sum():,}')
display(test_only_locations.sort_values(LOCATION_COLUMNS).head(100))

Test-only TOWN + BLOCK + STREET locations: 50
Of these, direct HDB matches: 50


,TOWN,BLOCK,STREET,TOWN_KEY,BLOCK_KEY,STREET_KEY,LATITUDE,LONGITUDE,MAX_FLOOR,YEAR_COMPLETED,REGION,has_hdb_match
33,ang mo kio,228a,ang mo kio street 23,ang mo kio,228a,ang mo kio street 23,1.368035,103.839426,27,2020,north-east region,True
12,ang mo kio,340,Ang Mo Kio Avenue 1,ang mo kio,340,ang mo kio avenue 1,1.364450,103.849464,2,1977,north-east region,True
44,bedok,154a,bedok south road,bedok,154a,bedok south road,1.318183,103.946754,18,2021,east region,True
39,bedok,547a,bedok north avenue 1,bedok,547a,bedok north avenue 1,1.329939,103.926093,16,2020,east region,True
8,bishan,287,bishan street 24,bishan,287,bishan street 24,1.357731,103.843785,4,1991,central region,True
9,bishan,452,Sin Ming Avenue,bishan,452,sin ming avenue,1.358575,103.829748,4,1988,central region,True
1,bukit batok,261,bukit batok east avenue 4,bukit batok,261,bukit batok east avenue 4,1.350529,103.758806,4,1984,west region,True
15,bukit batok,440c,Bukit Batok West Avenue 8,bukit batok,440c,bukit batok west avenue 8,1.355138,103.737861,14,2017,west region,True
7,bukit batok,443d,bukit batok west avenue 8,bukit batok,443d,bukit batok west avenue 8,1.355593,103.737118,14,2017,west region,True
19,bukit batok,464b,bukit batok west avenue 8,bukit batok,464b,bukit batok west avenue 8,1.357997,103.742102,17,2020,west region,True
